# 🚗 AVL GenAI Engineering Assistant — RAG Pipeline PoC
**Author:** Sanusi Isiaka Olatunji | M.Sc. Data Science | University of Leoben
**Aligned with:** AVL List GmbH — Working Student: Generative AI & Data Engineering
**AI Engine:** Google Gemini API (free tier) + FAISS vector store

---

### Pipeline stages covered:
| Stage | AVL Job Requirement | Implementation |
|---|---|---|
| 1️⃣ Data ingestion | Data & indexing preparation | Document loader + chunker |
| 2️⃣ Embeddings | Embedding & vector storage | Sentence-transformers + FAISS |
| 3️⃣ Retrieval | RAG retrieval pipeline | Cosine similarity top-k search |
| 4️⃣ Generation | GenAI chat/assistance system | Gemini 1.5 Flash (free) |
| 5️⃣ Evaluation | Quality metrics & monitoring | Faithfulness, relevance, drift |
| 6️⃣ Logging | Logging & drift monitoring | CSV audit log + analytics dashboard |
| 7️⃣ UI | Delivery to business units | Gradio web interface |

In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
!pip install -q google-generativeai sentence-transformers faiss-cpu gradio pandas matplotlib
print('✅ All dependencies installed')
print('   google-generativeai   — Gemini free API')
print('   sentence-transformers — local embeddings (no API cost)')
print('   faiss-cpu             — vector store & similarity search')
print('   gradio                — web UI')

In [ ]:
# ── CELL 2: API Key (Colab Secrets) ──────────────────────────────────────────
import google.generativeai as genai
import os
from google.colab import userdata

# Configure the API key using the environment variable
os.environ['Gemini_API_Key'] = userdata.get('Gemini_API_Key')
genai.configure(api_key=os.environ['Gemini_API_Key'])

print("Available models that support `generateContent`:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

In [ ]:
# ── CELL 3: Imports & configuration ──────────────────────────────────────────
import os, csv, json, time, datetime, textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

LOG_FILE   = 'rag_query_log.csv'
EVAL_FILE  = 'rag_eval_log.csv'
MODEL      = 'models/gemini-flash-latest'
EMBED_MODEL = 'all-MiniLM-L6-v2'   # fast, free, runs locally
TOP_K       = 3                     # chunks retrieved per query
CHUNK_SIZE  = 200                   # words per chunk

SYSTEM_PROMPT = """You are an expert engineering assistant for AVL List GmbH,
specializing in automotive testing, powertrain development, and data engineering.

Answer questions using ONLY the context provided below.
If the answer is not in the context, say: 'I could not find this in the knowledge base.'
Be concise (3-4 sentences), precise, and cite which document section your answer came from.
Always end with one follow-up question relevant to the topic."""

print(f'✅ Configuration loaded')
print(f'   LLM model   : {MODEL}')
print(f'   Embed model : {EMBED_MODEL} (runs locally — free)')
print(f'   Top-K chunks: {TOP_K}')
print(f'   Chunk size  : {CHUNK_SIZE} words')

In [ ]:
# ── CELL 4: Knowledge base — AVL engineering documents ───────────────────────
# Simulates real company documents (manuals, guidelines, reports)
# In production: replace with actual PDFs / SharePoint / database content

DOCUMENTS = [
    {
        'title': 'AVL PUMA Test System — User Manual',
        'content': """
        The AVL PUMA Open test system is a modular automation platform for engine and powertrain testing.
        It provides real-time control of test cell equipment including dynamometers, throttle actuators,
        fuel measurement systems, and emission analyzers. PUMA supports deterministic test execution
        with cycle times down to 1 millisecond. The system uses a client-server architecture where
        the PUMA server runs on a real-time operating system and the client provides the user interface.
        Data acquisition is handled through a distributed I/O system supporting analog, digital,
        and fieldbus signals. PUMA integrates with AVL CONCERTO for post-processing and analysis.
        Test sequences can be programmed using the built-in scripting language or imported from
        external cycle files in standard formats like NEDC, WLTP, and RDE drive cycles.
        Calibration data from ECU flashing tools like AVL CRUISE can be directly linked to test runs.
        The system logs all measurements in a proprietary binary format with timestamps at microsecond resolution.
        """
    },
    {
        'title': 'Emission Testing Standards — WLTP & RDE Guide',
        'content': """
        The Worldwide Harmonised Light Vehicle Test Procedure (WLTP) replaced the NEDC cycle in 2017
        for CO2 and fuel consumption measurements in Europe. WLTP uses a four-phase cycle: Low, Medium,
        High, and Extra High, covering speeds from 0 to 131 km/h over 30 minutes and 23.26 km.
        Real Driving Emissions (RDE) testing complements WLTP by measuring actual on-road emissions
        using Portable Emissions Measurement Systems (PEMS). RDE tests must be conducted on public
        roads under defined boundary conditions including altitude below 700 m, ambient temperature
        between 0 and 30 degrees Celsius, and trip duration between 90 and 120 minutes.
        NOx conformity factors define the maximum ratio of on-road to laboratory emissions.
        AVL provides specialized PEMS equipment and data analysis software for RDE compliance testing.
        The Not-to-Exceed (NTE) zone concept defines operating regions where emission limits apply.
        Cold start emissions are weighted and included in the final RDE result calculation.
        """
    },
    {
        'title': 'Data Engineering Guide — AVL Testbed Data Pipeline',
        'content': """
        AVL testbed data pipelines handle high-frequency measurement data from engine test cells.
        Raw data ingestion occurs at rates up to 100 kHz for combustion analysis signals.
        The ETL pipeline consists of three stages: extraction from proprietary measurement formats,
        transformation including unit conversion, filtering, and resampling, and loading into
        the central data lake built on Azure Data Lake Storage Gen2.
        Metadata tagging uses a standardized schema including test ID, engine variant, operator ID,
        ambient conditions, and calibration version for full traceability.
        Apache Spark is used for large-scale batch processing of historical test data.
        Real-time streaming uses Apache Kafka with custom connectors for PUMA data sources.
        The data catalog is maintained in Azure Purview for discoverability and governance.
        Quality checks include range validation, sensor drift detection, and cross-channel plausibility.
        Data retention policies follow ISO 17025 laboratory accreditation requirements.
        """
    },
    {
        'title': 'GenAI Integration Guide — AVL Internal AI Policy',
        'content': """
        AVL's GenAI integration framework governs the deployment of large language models
        in engineering and business workflows. All GenAI applications must undergo a risk
        assessment before production deployment, classifying outputs as informational,
        decision-support, or autonomous action categories.
        Retrieval-Augmented Generation (RAG) is the preferred architecture for knowledge-based
        assistants, ensuring responses are grounded in verified company documentation.
        Hallucination monitoring is mandatory: all LLM outputs are evaluated for faithfulness
        to source documents using automated scoring pipelines.
        On-premise deployment using open-source models is required for data classified as
        confidential or restricted. Cloud deployments use Azure OpenAI Service within the
        EU data boundary. Vector databases approved for production use include pgvector,
        Azure AI Search, and FAISS for prototyping.
        Prompt injection attacks must be mitigated through input validation and output filtering.
        Model drift monitoring tracks response quality over time using a set of golden queries.
        """
    },
    {
        'title': 'Powertrain Development — Hybrid & EV Testing',
        'content': """
        Modern powertrain development at AVL covers combustion engines, hybrid systems,
        battery electric vehicles (BEV), and fuel cell electric vehicles (FCEV).
        Hardware-in-the-Loop (HiL) simulation allows testing of electronic control units
        without physical powertrain components, accelerating development cycles.
        Battery testing includes capacity measurement, impedance spectroscopy, and
        cycle life testing under controlled temperature conditions.
        The AVL E-Storage system provides battery simulation for HiL and SiL test environments.
        Electric motor characterization covers efficiency maps, torque ripple analysis,
        and thermal management validation across the full operating range.
        NVH (Noise, Vibration, and Harshness) analysis for electric drivetrains focuses
        on tonal noise from inverters and gear whine from transmission systems.
        Over-the-Air (OTA) update validation is an emerging test discipline requiring
        specialized cybersecurity and functional safety test procedures.
        """
    },
    {
        'title': 'Machine Learning for Engine Calibration — Technical Report',
        'content': """
        Machine learning has become central to engine calibration at AVL, reducing
        measurement effort by up to 80% compared to traditional Design of Experiments (DoE).
        Gaussian Process Regression (GPR) models are used to build virtual engine maps
        from sparse measurement data, enabling interpolation across the full operating range.
        The AVL CAMEO software implements model-based calibration workflows integrating
        GPR models with multi-objective optimization algorithms.
        Deep learning models, particularly CNNs and LSTMs, are applied to in-cylinder
        pressure trace analysis for combustion diagnostics and knock detection.
        Transfer learning enables rapid adaptation of base models to new engine variants,
        significantly reducing training data requirements.
        Federated learning approaches are being explored for collaborative model training
        across multiple OEM customers without sharing proprietary engine data.
        Anomaly detection models monitor sensor data in real time to flag measurement
        issues or unexpected engine behavior during testing.
        """
    },
]

print(f'✅ Knowledge base loaded — {len(DOCUMENTS)} documents')
for d in DOCUMENTS:
    words = len(d['content'].split())
    print(f'   📄 {d["title"]} ({words} words)')

In [ ]:
# ── CELL 5: Stage 1 — Data ingestion & chunking ───────────────────────────────
print('🔄 Stage 1: Ingestion & Chunking...')

def chunk_document(doc, chunk_size=CHUNK_SIZE, overlap=40):
    """Split document into overlapping word chunks for better retrieval."""
    words  = doc['content'].split()
    chunks = []
    start  = 0
    while start < len(words):
        end   = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        chunks.append({
            'chunk_id'  : f"{doc['title'][:30].replace(' ','_')}_chunk{len(chunks)}",
            'source'    : doc['title'],
            'content'   : chunk,
            'word_count': len(chunk.split()),
            'start_word': start,
        })
        start += chunk_size - overlap
    return chunks

all_chunks = []
for doc in DOCUMENTS:
    chunks = chunk_document(doc)
    all_chunks.extend(chunks)
    print(f'   📄 "{doc["title"][:45]}" → {len(chunks)} chunks')

print(f'\n✅ Ingestion complete')
print(f'   Total chunks : {len(all_chunks)}')
print(f'   Avg words/chunk: {np.mean([c["word_count"] for c in all_chunks]):.0f}')

In [ ]:
# ── CELL 6: Stage 2 — Embeddings & FAISS vector index ───────────────────────
print('🔄 Stage 2: Embedding & Indexing (this takes ~30s first run)...')
from sentence_transformers import SentenceTransformer
import faiss

# Load local embedding model (no API cost)
embedder = SentenceTransformer(EMBED_MODEL)

# Embed all chunks
texts      = [c['content'] for c in all_chunks]
t0         = time.time()
embeddings = embedder.encode(texts, show_progress_bar=True,
                              convert_to_numpy=True, normalize_embeddings=True)
embed_time = time.time() - t0

# Build FAISS index (Inner Product = cosine similarity on normalised vectors)
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype('float32'))

print(f'\n✅ Vector index built')
print(f'   Embedding model : {EMBED_MODEL}')
print(f'   Vector dimension: {dim}')
print(f'   Vectors indexed : {index.ntotal}')
print(f'   Embedding time  : {embed_time:.1f}s')

In [ ]:
# ── CELL 7: Stage 3 & 4 — Retrieval + Generation (full RAG pipeline) ────────
print('🔄 Stage 3-4: RAG pipeline functions ready...')
import google.generativeai as genai
genai.configure(api_key=os.environ['Gemini_API_Key'])
gemini = genai.GenerativeModel(MODEL, system_instruction=SYSTEM_PROMPT)

def retrieve(query, top_k=TOP_K):
    """Stage 3: Embed query and retrieve top-k similar chunks from FAISS."""
    q_vec = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        chunk = all_chunks[idx].copy()
        chunk['similarity'] = float(score)
        results.append(chunk)
    return results

def build_context(chunks):
    """Build context string from retrieved chunks with source attribution."""
    parts = []
    for i, c in enumerate(chunks, 1):
        parts.append(f"[Source {i}: {c['source']}]\n{c['content']}")
    return '\n\n'.join(parts)

def generate(query, context, chat_history=None):
    """Stage 4: Call Gemini with query + retrieved context."""
    prompt = f"Context:\n{context}\n\nQuestion: {query}"
    t0 = time.time()
    try:
        resp = gemini.generate_content(prompt)
        return resp.text.strip(), time.time() - t0
    except Exception as e:
        return f'Generation error: {e}', 0.0

def log_query(query, chunks, answer, elapsed, eval_scores):
    """Log every RAG interaction for audit & monitoring."""
    exists = Path(LOG_FILE).exists()
    with open(LOG_FILE, 'a', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        if not exists:
            w.writerow(['timestamp','query','top_source','similarity',
                        'answer_preview','response_time_s',
                        'faithfulness','relevance','completeness'])
        w.writerow([
            datetime.datetime.now().isoformat(),
            query[:100],
            chunks[0]['source'][:50] if chunks else 'none',
            round(chunks[0]['similarity'], 4) if chunks else 0,
            answer[:150],
            round(elapsed, 2),
            eval_scores.get('faithfulness', 0),
            eval_scores.get('relevance', 0),
            eval_scores.get('completeness', 0),
        ])

def rag(query, verbose=True):
    """Full RAG pipeline: retrieve → build context → generate → evaluate → log."""
    # Retrieve
    chunks  = retrieve(query)
    context = build_context(chunks)
    # Generate
    answer, elapsed = generate(query, context)
    # Evaluate (Stage 5 — called inline)
    scores = evaluate(query, answer, chunks)
    # Log
    log_query(query, chunks, answer, elapsed, scores)
    if verbose:
        print(f'\n❓ Query: {query}')
        print(f'\n📚 Retrieved chunks:')
        for c in chunks:
            print(f'   [{c["similarity"]:.3f}] {c["source"]}')
        print(f'\n🤖 Answer:')
        print(textwrap.fill(answer, width=70))
        print(f'\n📊 Eval scores: Faithfulness={scores["faithfulness"]:.2f} '
              f'| Relevance={scores["relevance"]:.2f} '
              f'| Completeness={scores["completeness"]:.2f}')
        print(f'⏱  {elapsed:.2f}s')
        print('─'*70)
    return answer, chunks, scores

print('✅ RAG pipeline ready')
print('   retrieve(query)  → FAISS top-k similarity search')
print('   generate(...)    → Gemini LLM with grounded context')
print('   rag(query)       → full pipeline in one call')

In [ ]:
# ── CELL 8: Stage 5 — Evaluation & quality metrics ───────────────────────────
print('🔄 Stage 5: Evaluation functions...')

def evaluate(query, answer, chunks):
    """
    Lightweight RAG evaluation — no extra API calls needed.

    Faithfulness  : Are answer keywords found in retrieved context?
    Relevance     : Does context semantically match the query?
    Completeness  : Is the answer long enough to be useful?
    """
    context_text = ' '.join([c['content'].lower() for c in chunks])
    answer_lower = answer.lower()
    query_lower  = query.lower()

    # Faithfulness: fraction of answer words found in context
    answer_words = [w for w in answer_lower.split() if len(w) > 4]
    if answer_words:
        found = sum(1 for w in answer_words if w in context_text)
        faithfulness = min(found / len(answer_words) * 1.5, 1.0)
    else:
        faithfulness = 0.0

    # Relevance: fraction of query keywords found in top chunk
    query_words = [w for w in query_lower.split() if len(w) > 3]
    top_chunk   = chunks[0]['content'].lower() if chunks else ''
    if query_words:
        hits = sum(1 for w in query_words if w in top_chunk)
        relevance = min(hits / len(query_words) * 2.0, 1.0)
    else:
        relevance = 0.0

    # Completeness: answer length score (50-300 words = ideal)
    word_count   = len(answer.split())
    completeness = min(word_count / 80, 1.0) if word_count < 80 else max(1.0 - (word_count - 80) / 300, 0.6)

    return {
        'faithfulness' : round(faithfulness, 3),
        'relevance'    : round(relevance, 3),
        'completeness' : round(completeness, 3),
        'overall'      : round((faithfulness + relevance + completeness) / 3, 3),
    }

# Drift monitoring — compare current scores vs historical baseline
def check_drift(current_scores, threshold=0.15):
    """Flag if current eval scores drop significantly below historical average."""
    if not Path(LOG_FILE).exists():
        return False, {}
    df = pd.read_csv(LOG_FILE)
    if len(df) < 3:
        return False, {}
    baseline = {
        'faithfulness' : df['faithfulness'].mean(),
        'relevance'    : df['relevance'].mean(),
        'completeness' : df['completeness'].mean(),
    }
    drift_flags = {
        k: (baseline[k] - current_scores[k]) > threshold
        for k in baseline
    }
    has_drift = any(drift_flags.values())
    return has_drift, drift_flags

print('✅ Evaluation metrics ready')
print('   Faithfulness  — answer grounded in retrieved context')
print('   Relevance     — context matches query intent')
print('   Completeness  — answer length and substance')
print('   Drift monitor — flags quality drops vs historical baseline')

In [ ]:
# ── CELL 9: Run Demo — 6 engineering queries ──────────────────────────────────
DEMO_QUERIES = [
    'What is the AVL PUMA system and what test cycles does it support?',
    'What are the boundary conditions for a valid RDE test?',
    'How does AVL use Apache Kafka in its data pipeline?',
    'What vector databases are approved for GenAI production use at AVL?',
    'How is machine learning used for engine calibration at AVL?',
    'What is Hardware-in-the-Loop testing and why is it used for EVs?',
]

print('='*70)
print('  RAG PIPELINE DEMO — 6 Engineering Queries')
print('='*70)

all_scores = []
for q in DEMO_QUERIES:
    answer, chunks, scores = rag(q)
    all_scores.append(scores)
    # Drift check
    has_drift, drift_flags = check_drift(scores)
    if has_drift:
        print(f'  ⚠️  DRIFT DETECTED: {drift_flags}')

avg = {k: round(np.mean([s[k] for s in all_scores]), 3)
       for k in ['faithfulness','relevance','completeness','overall']}

print('\n' + '='*70)
print('  EVALUATION SUMMARY')
print('='*70)
print(f'  Queries run      : {len(DEMO_QUERIES)}')
print(f'  Avg Faithfulness : {avg["faithfulness"]}')
print(f'  Avg Relevance    : {avg["relevance"]}')
print(f'  Avg Completeness : {avg["completeness"]}')
print(f'  Avg Overall      : {avg["overall"]}')
print('='*70)

In [ ]:
# ── CELL 10: Ask your own question ───────────────────────────────────────────
# Change MY_QUERY and re-run this cell

MY_QUERY = "How does AVL handle data quality checks in its testbed pipeline?"

answer, chunks, scores = rag(MY_QUERY)

In [ ]:
# ── CELL 11: Analytics dashboard ─────────────────────────────────────────────
CB='#0d1117'; CP='#161b22'; CBR='#30363d'
CT='#e6edf3'; CM='#8b949e'; CN='#58a6ff'; CA='#f85149'; CG='#3fb950'; CY='#e3b341'

plt.rcParams.update({
    'figure.facecolor':CB,'axes.facecolor':CP,'axes.edgecolor':CBR,
    'axes.labelcolor':CT,'xtick.color':CM,'ytick.color':CM,
    'text.color':CT,'grid.color':CBR,'grid.linestyle':'--','grid.linewidth':0.5,
    'font.family':'monospace'
})

df = pd.read_csv(LOG_FILE)
df['timestamp'] = pd.to_datetime(df['timestamp'])

fig = plt.figure(figsize=(18, 12), facecolor=CB)
fig.suptitle(
    'AVL GenAI RAG Pipeline — Query Analytics & Evaluation Dashboard\n'
    'Sanusi Isiaka Olatunji  |  M.Sc. Data Science  |  University of Leoben',
    fontsize=13, color=CT, fontweight='bold', y=0.98
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

x = range(len(df))

# (A) Eval scores over time
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(x, df['faithfulness'],  color=CG, lw=2, marker='o', ms=5, label='Faithfulness')
ax1.plot(x, df['relevance'],     color=CN, lw=2, marker='s', ms=5, label='Relevance')
ax1.plot(x, df['completeness'],  color=CY, lw=2, marker='^', ms=5, label='Completeness')
ax1.axhline(0.7, color=CA, lw=1, ls='--', label='Quality threshold (0.7)')
ax1.set_title('RAG Evaluation Scores per Query', fontsize=10, color=CT)
ax1.set_xlabel('Query #', fontsize=8); ax1.set_ylabel('Score (0–1)', fontsize=8)
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=8, framealpha=0.2, labelcolor=CT); ax1.grid(True, alpha=0.4)

# (B) Similarity score distribution
ax2 = fig.add_subplot(gs[0, 2])
ax2.bar(x, df['similarity'], color=CN, alpha=0.8)
ax2.axhline(df['similarity'].mean(), color=CG, lw=1.5, ls='--',
            label=f"Mean: {df['similarity'].mean():.3f}")
ax2.set_title('Top-Chunk Similarity Score', fontsize=10, color=CT)
ax2.set_xlabel('Query #', fontsize=8); ax2.set_ylabel('Cosine Similarity', fontsize=8)
ax2.legend(fontsize=8, framealpha=0.2, labelcolor=CT); ax2.grid(True, alpha=0.4)

# (C) Source document retrieval frequency
ax3 = fig.add_subplot(gs[1, :2])
source_counts = df['top_source'].value_counts()
short_labels  = [s[:35] + '...' if len(s) > 35 else s for s in source_counts.index]
bars = ax3.barh(short_labels, source_counts.values, color=CY, alpha=0.8, height=0.6)
ax3.set_title('Most Retrieved Document Sources', fontsize=10, color=CT)
ax3.set_xlabel('Times retrieved as top result', fontsize=8)
ax3.tick_params(axis='y', labelsize=7.5); ax3.grid(True, axis='x', alpha=0.4)
for bar, val in zip(bars, source_counts.values):
    ax3.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=8, color=CT)

# (D) Summary stats box
ax4 = fig.add_subplot(gs[1, 2])
ax4.axis('off')
summary = [
    f'Total queries      : {len(df)}',
    f'Avg response time  : {df["response_time_s"].mean():.2f}s',
    f'Avg similarity     : {df["similarity"].mean():.3f}',
    f'Avg faithfulness   : {df["faithfulness"].mean():.3f}',
    f'Avg relevance      : {df["relevance"].mean():.3f}',
    f'Avg completeness   : {df["completeness"].mean():.3f}',
    '',
    f'Vector store       : FAISS',
    f'Embed model        : {EMBED_MODEL}',
    f'LLM                : {MODEL}',
    f'Chunks indexed     : {index.ntotal}',
    f'Top-K retrieval    : {TOP_K}',
    f'Cost               : FREE ✅',
]
ax4.text(0.05, 0.95, '\n'.join(summary), transform=ax4.transAxes,
         fontsize=8, color=CT, va='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor=CP, edgecolor=CBR, alpha=0.9))
ax4.set_title('Pipeline Summary', fontsize=10, color=CT)

plt.savefig('avl_rag_dashboard.png', dpi=150, bbox_inches='tight', facecolor=CB)
plt.show()
print('✅ Dashboard saved → avl_rag_dashboard.png')

In [ ]:
# ── CELL 12: Gradio Web UI ────────────────────────────────────────────────────
import gradio as gr

EXAMPLES = [
    'What is the AVL PUMA system and what test cycles does it support?',
    'What are the RDE boundary conditions for ambient temperature?',
    'Which vector databases are approved for production GenAI at AVL?',
    'How does machine learning reduce measurement effort in calibration?',
    'What is HiL testing and how is it used for electric vehicles?',
    'How does AVL use Apache Kafka in its data engineering pipeline?',
]

def gradio_rag(message, chat_hist):
    if not message.strip():
        return '', chat_hist, ''
    answer, chunks, scores = rag(message, verbose=False)
    sources = '\n'.join([f"[{i+1}] {c['source']} (sim={c['similarity']:.3f})"
                         for i, c in enumerate(chunks)])
    eval_str = (f"Faithfulness: {scores['faithfulness']:.2f}  |  "
                f"Relevance: {scores['relevance']:.2f}  |  "
                f"Completeness: {scores['completeness']:.2f}  |  "
                f"Overall: {scores['overall']:.2f}")
    chat_hist.append((message, answer))
    info = f"📚 Sources retrieved:\n{sources}\n\n📊 Eval: {eval_str}"
    return '', chat_hist, info

with gr.Blocks(
    title='AVL GenAI RAG Assistant',
    theme=gr.themes.Base(primary_hue='blue'),
    css="""
    .header{text-align:center;padding:18px;background:#161b22;
            border-radius:12px;border:1px solid #30363d;margin-bottom:14px;}
    .header h1{color:#58a6ff;font-size:1.4em;margin:0;font-family:monospace;}
    .header p{color:#8b949e;margin:5px 0 0;font-size:0.83em;}
    .badge{display:inline-block;background:#1a3a1a;color:#3fb950;
           padding:2px 10px;border-radius:20px;font-size:0.78em;margin-top:5px;}
    """
) as demo:
    gr.HTML("""
    <div class='header'>
      <h1>🚗 AVL GenAI Engineering Assistant</h1>
      <p>RAG Pipeline · FAISS Vector Store · Gemini Flash Latest · Real-time Evaluation</p>
      <p><span class='badge'>✅ Retrieval-Augmented Generation — Grounded Answers Only</span></p>
      <p style='color:#8b949e;font-size:0.78em;margin-top:4px;'>
        Sanusi Isiaka Olatunji · M.Sc. Data Science · University of Leoben
      </p>
    </div>""")

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=380, label='RAG Assistant',
                                  bubble_full_width=False)
            msg  = gr.Textbox(placeholder='Ask an AVL engineering question...',
                               label='Query', lines=2)
            with gr.Row():
                send  = gr.Button('Send 🚀', variant='primary', scale=3)
                clear = gr.Button('Clear 🗑️', scale=1)
        with gr.Column(scale=1):
            info_box = gr.Textbox(label='📚 Sources & Evaluation Scores',
                                   lines=12, interactive=False)

    gr.Examples(EXAMPLES, inputs=msg, label='💡 Example queries')

    send.click(gradio_rag,  [msg, chatbot], [msg, chatbot, info_box])
    msg.submit(gradio_rag,  [msg, chatbot], [msg, chatbot, info_box])
    clear.click(lambda: ([], '', ''), None, [chatbot, msg, info_box])

demo.launch(share=True)

In [ ]:
# ── CELL 13: Download all outputs ────────────────────────────────────────────
from google.colab import files

for f in [LOG_FILE, 'avl_rag_dashboard.png']:
    if os.path.exists(f):
        files.download(f)
        print(f'⬇️  Downloaded: {f}')
    else:
        print(f'⚠️  Not found (run earlier cells first): {f}')